# 01 — Data layer

**Phase 1 deliverable:** three cached Parquet files and a printed quality report for each.

The notebooks in this project are *outputs*, not sources. Every line of logic lives in
`src/stock_retrofit/`; these cells only call it. That keeps the package testable and keeps
the notebooks readable.

> **yfinance is the primary source in this build.** Not by preference — the Settrade Open API is
> credential-gated behind a broker relationship and no credentials exist in this environment.
> Spec R2 explicitly provides for this fallback. See `docs/settrade-api-notes.md` for what was
> tried and what remains open.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("stock-retrofit @", ROOT)

## Fetch and cache

The only network path in the package. Everything downstream reads the Parquet cache (spec R3).
Each write lands a `.meta.json` sidecar recording source, timestamp, row count, date range and a
content hash, so any result traces back to the exact bytes that produced it.

In [ ]:
from datetime import date
from stock_retrofit.data import fetch

metas = fetch(["KBANK", "SCB", "BAY"], source="yfinance", start=date(2000, 1, 1))
for symbol, meta in metas.items():
    print(f"{symbol:6s} {meta.rows:5d} bars  {meta.start} .. {meta.end}  "
          f"hash {meta.content_hash[:12]}  repairs {meta.repairs['count']}")

## Quality gates

Two tiers, deliberately. **Structural violations raise** — `high < low`, a close outside
`[low, high]`, duplicate dates, or an *unexplained* move beyond SET's ±30% band. None of those are
possible in correct data. **Advisories are reported** — missing sessions, zero-volume days. Those
are real facts about a thin market, not corruption.

In [ ]:
from stock_retrofit.data import quality_report

for symbol in ["KBANK", "SCB", "BAY"]:
    print(quality_report(symbol).render())
    print()

### What the gate found, and why each is handled the way it is

**Vendor bar defects.** 3 bars in KBANK, 1 in SCB, 6 in BAY have a `close` outside `[low, high]`
by one to three ticks. Repair is a *separate stage* from the gate: the gate still raises on
anything handed to it unrepaired, and every repair is recorded in the meta sidecar. A bad bar
being fixed on the record is fine; a bad bar passing silently is the thing this layer exists to
prevent.

**SCB's 4-session gap and +39.9% move.** Trading halted 2022-04-21 → 2022-04-26 around the issuer
substitution, and the price discontinuity lands on 2022-04-27, the first session back. The gate
treats a move that *spans* a registered break as explained rather than as a limit violation —
matching on the break date alone would have missed it.

In [ ]:
import json
from stock_retrofit.paths import RAW_DIR

print(json.dumps(json.loads((RAW_DIR / "BAY.meta.json").read_text())["repairs"], indent=2)[:1200])

## Instrument semantics

The SCB caveat lives in code, not in someone's memory (spec R13–R15). A caller can never receive a
series spanning a registered break without a signal that it did.

In [ ]:
from stock_retrofit.data import describe

for symbol in ["KBANK", "SCB", "BAY"]:
    print(describe(symbol)); print()

In [ ]:
from stock_retrofit.data import load

truncated = load("SCB")                                   # default policy
flagged   = load("SCB", policy="full_with_changepoint")
print("truncate_at_break     :", len(truncated), "bars from", truncated["date"].min().date())
print("full_with_changepoint :", len(flagged), "bars, changepoints flagged:",
      int(flagged["is_changepoint"].sum()))

**An empirical finding worth recording:** Yahoo's `SCB.BK` series *begins* 2022-04-20 — the
vendor already carries SCBX only. So `truncate_at_break` is satisfied trivially and
`full_with_changepoint` cannot be populated from this source; there is no pre-break history to
keep. The policy flag and registry entry are implemented and tested regardless, because the
caveat belongs in code and because a future Settrade fetch could supply the missing years.

## Reconciliation

Spec R10 wants two independent sources cross-checked, never averaged. With only one source
reachable this degrades to a cache-vs-vendor check — which catches staleness and vendor revisions
but *cannot* catch an error Yahoo makes consistently. The output labels the actual sources so the
limitation is visible rather than implied.

In [ ]:
from stock_retrofit.data import reconcile

table = reconcile("KBANK", against="yfinance")
print(f"{len(table)} overlapping dates, "
      f"{int(table['exceeds_one_tick'].sum())} exceed one tick of disagreement")
table.tail()